# 🔬 Ablation Study — Kaggle Training (Step 5/8)
**Trains all ablation configurations sequentially to generate Tables 4 & 5.**

## ⚙️ Setup
1. **Datasets** — Add all 12 datasets (Input → Add Data)
2. **Secret** — Add `HF_TOKEN` secret
3. **Accelerator** — GPU T4 x2
4. **Run All**

> Note: Like the baselines notebook, this automatically resumes from the last completed epoch of whichever configuration it was running if the 12h session expires.

**Sequence**: Tiny → Base → Large → Baselines → **Ablation** → PaperEvals → LOGO → Outputs

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 0: Clone Repo + Install Dependencies
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path

REPO_URL  = "https://github.com/MIHMahmudEli/ai-image-detection-research.git"
CLONE_DIR = Path("/kaggle/working/ai-image-detection-research")

if not CLONE_DIR.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--rebase"], check=False)

os.chdir(str(CLONE_DIR))\nsys.path.insert(0, str(CLONE_DIR / "model"))

subprocess.run([sys.executable, "-m", "pip", "install",
    "huggingface_hub", "open_clip_torch", "scipy", "scikit-learn",
    "-q", "--disable-pip-version-check"], check=False)

print(f"Project root: {CLONE_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Imports & Environment
# ═══════════════════════════════════════════════════════════
import os, sys, math, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path("/kaggle/working/ai-image-detection-research/model")))
from src.kaggle_utils import KaggleEnv

env = KaggleEnv(project_root_search=True)
PROJECT_ROOT = env.project_root
os.chdir(env.working_dir)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE_TEST = not torch.cuda.is_available()
print(f"Device: {device} | SMOKE: {SMOKE_TEST}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: Config & Dataloaders
# ═══════════════════════════════════════════════════════════
from src.dataset import create_split_dataloaders
from src.config import Config

cfg = Config()
NUM_EPOCHS  = 1   if SMOKE_TEST else 20
IMAGE_SIZE  = 224 if SMOKE_TEST else 384
BATCH_SIZE  = 8   if SMOKE_TEST else 64
NUM_WORKERS = 0   if SMOKE_TEST else 4
MAX_SAMPLES = 600 if SMOKE_TEST else None

cfg.training.epochs     = NUM_EPOCHS
cfg.training.image_size = IMAGE_SIZE
cfg.training.batch_size = BATCH_SIZE

_manifest = PROJECT_ROOT / "dataset" / "metadata" / "train_manifest.csv"
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.download_manifest(_manifest)
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.rebuild_manifest_from_kaggle(_manifest)
if _manifest.exists(): cfg.dataset.metadata_paths = [str(_manifest)]

_split_name = "split_indices_smoke.json" if SMOKE_TEST else "split_indices.json"
_split_path = PROJECT_ROOT / "dataset" / "metadata" / _split_name
if not _split_path.exists() and not SMOKE_TEST and env.hf_token:
    try:
        from huggingface_hub import hf_hub_download
        import shutil
        _dl = hf_hub_download(repo_id=env.hf_manifest_repo, filename="split_indices.json", repo_type="model", token=env.hf_token)
        shutil.copy2(_dl, _split_path)
    except Exception as e: print(f"Warning: could not download split_indices from HF: {e}")

train_loader, val_loader, test_loader = create_split_dataloaders(
    root_dir=str(PROJECT_ROOT), metadata_paths=[str(_manifest)],
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, size=IMAGE_SIZE,
    val_split=0.10, test_split=0.10, seed=SEED, use_weighted_sampler=True,
    split_index_path=str(_split_path), max_samples=MAX_SAMPLES
)
print(f"Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: Ablation Configurations
# ═══════════════════════════════════════════════════════════
from src.model import build_mfft, count_parameters

ABLATIONS = [
    {"name": "spatial_only", "label": "Spatial-only (no bands)", "table": 4, "ablation": {"spatial_only": True, "use_fga": False, "fusion_mode": "concat"}},
    {"name": "band_low", "label": "Low-frequency only", "table": 4, "ablation": {"skip_bands": ["mid", "high"], "use_fga": False, "fusion_mode": "concat"}},
    {"name": "band_mid", "label": "Mid-frequency only", "table": 4, "ablation": {"skip_bands": ["low", "high"], "use_fga": False, "fusion_mode": "concat"}},
    {"name": "band_high", "label": "High-frequency only", "table": 4, "ablation": {"skip_bands": ["low", "mid"], "use_fga": False, "fusion_mode": "concat"}},
    {"name": "fusion_concat", "label": "All bands (concat)", "table": 4, "ablation": {"fusion_mode": "concat", "use_fga": False}},
    {"name": "fusion_avg", "label": "All bands (avg)", "table": 4, "ablation": {"fusion_mode": "avg", "use_fga": False}},
    {"name": "fusion_max", "label": "All bands (max)", "table": 4, "ablation": {"fusion_mode": "max", "use_fga": False}},
    {"name": "no_fga", "label": "All bands (cross-attn, no FGA)", "table": 4, "ablation": {"fusion_mode": "attention", "use_fga": False}},
    {"name": "bands_2", "label": "2 bands (low, high)", "table": 5, "variant": "base", "ablation": {"skip_bands": ["mid"], "fusion_mode": "attention", "use_fga": True}},
    {"name": "bands_4", "label": "4 bands (low, low-mid, mid-high, high)", "table": 5, "variant": "large", "ablation": {"fusion_mode": "attention", "use_fga": True}},
    {"name": "upgrade_pretrained", "label": "MFFT + pretrained extractors", "table": 6, "kwargs": {"pretrained_extractors": True}},
    {"name": "upgrade_npr", "label": "MFFT + NPR high-band residual", "table": 6, "kwargs": {"use_npr": True}},
    {"name": "upgrade_soft", "label": "MFFT + soft band masks", "table": 6, "kwargs": {"soft_masks": True}},
    {"name": "upgrade_combined", "label": "MFFT + all upgrades", "table": 6, "kwargs": {"pretrained_extractors": True, "use_npr": True, "soft_masks": True}},
]

if SMOKE_TEST:
    _keep = {"spatial_only", "fusion_max", "no_fga", "upgrade_combined"}
    ABLATIONS = [a for a in ABLATIONS if a["name"] in _keep]
    print(f"SMOKE_TEST: running only {[a['name'] for a in ABLATIONS]}")
else:
    print(f"{len(ABLATIONS)} ablation experiments defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 4: Train Ablations Sequentially (with resume support)
# ═══════════════════════════════════════════════════════════
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

AMP = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=AMP)
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.training.label_smoothing)
RESULTS_DIR = PROJECT_ROOT / "paper" / "result" / "full_scale" / "ablation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

all_results = []

for ablation_cfg in ABLATIONS:
    name = ablation_cfg["name"]
    print(f"\n{'='*70}\nTraining {ablation_cfg['label']} ({name})\n{'='*70}")

    # Skip if already done
    res_file = RESULTS_DIR / f"{name}.json"
    if res_file.exists():
        print(f"{name} already completed. Skipping.")
        with open(res_file) as f: all_results.append(json.load(f))
        continue

    variant = ablation_cfg.get("variant", "base")
    model = build_mfft(variant, ablation=ablation_cfg.get("ablation"), **ablation_cfg.get("kwargs", {})).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=min(500, len(train_loader)))
    cosine = CosineAnnealingWarmRestarts(optimizer, T_0=NUM_EPOCHS*len(train_loader), T_mult=2, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[min(500, len(train_loader))])

    ckpt_dir = PROJECT_ROOT / "model" / "checkpoints" / f"ablation_{name}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    start_epoch = 0; best_acc = 0.0

    # Resume
    resume_ckpt = env.find_resume_checkpoint(ckpt_dir)
    if not resume_ckpt: resume_ckpt = env.download_latest_checkpoint(ckpt_dir, f"ablation_{name}")
    if resume_ckpt:
        state = env.load_checkpoint(resume_ckpt, model, optimizer, scheduler, scaler, device=device)
        if state:
            start_epoch, best_acc = state["epoch"] + 1, state["best_acc"]
            history = state["history"] or history
            print(f"  Resumed epoch {start_epoch}, best={best_acc:.2f}%")

    # Train loop
    for epoch in range(start_epoch, NUM_EPOCHS):
        model.train()
        total_loss = correct = total = 0
        pbar = tqdm(train_loader, desc=f"{name} Ep {epoch+1}/{NUM_EPOCHS}")
        for images, labels in pbar:
            try:
                images, labels = images.to(device), labels.to(device)
                with torch.amp.autocast("cuda", enabled=AMP):
                    logits = model(images); loss = criterion(logits, labels)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(); scheduler.step()
                total_loss += loss.item()
                correct += (logits.argmax(-1) == labels).sum().item(); total += labels.size(0)
                pbar.set_postfix({"loss": f"{total_loss/max(1,total//BATCH_SIZE):.4f}", "acc": f"{correct/total*100:.2f}%"})
            except Exception as e: print(f"  Skip batch: {e}"); optimizer.zero_grad()

        train_acc = correct / total * 100
        history["train_acc"].append(train_acc); history["train_loss"].append(total_loss / len(train_loader))

        model.eval(); val_loss = val_correct = val_total = 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"{name} Val", leave=False):
                try:
                    images, labels = images.to(device), labels.to(device)
                    logits = model(images); val_loss += criterion(logits, labels).item()
                    val_correct += (logits.argmax(-1) == labels).sum().item(); val_total += labels.size(0)
                except Exception: pass
        val_acc = val_correct / val_total * 100
        history["val_acc"].append(val_acc); history["val_loss"].append(val_loss / len(val_loader))
        print(f"  Ep {epoch+1}: train={train_acc:.2f}% val={val_acc:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc; torch.save(model.state_dict(), ckpt_dir / "best.pt")

        if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS - 1:
            ckpt_path = ckpt_dir / f"checkpoint_epoch_{epoch+1}.pt"
            env.save_checkpoint(ckpt_path, model, optimizer, scheduler, scaler, epoch, best_acc, history)
            env.upload_checkpoint(ckpt_path, f"ablation_{name}")

    # Test evaluation
    if start_epoch < NUM_EPOCHS:
        model.load_state_dict(torch.load(ckpt_dir / "best.pt", weights_only=True))
    model.eval(); test_correct = test_total = 0
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"{name} Test"):
            try:
                logits = model(images.to(device))
                test_correct += (logits.argmax(-1) == labels.to(device)).sum().item(); test_total += labels.size(0)
            except Exception: pass
    test_acc = test_correct / test_total * 100
    print(f"{name} done. Best val: {best_acc:.2f}%, Test: {test_acc:.2f}%")

    result = {
        "name": name, "label": ablation_cfg["label"], "table": ablation_cfg.get("table", 0),
        "params": count_parameters(model), "final_train_acc": round(train_acc, 2),
        "best_val_acc": round(best_acc, 2), "test_acc": round(test_acc, 2),
        "history": {k: [round(v, 4) for v in vs] for k, vs in history.items()}
    }
    all_results.append(result)
    with open(res_file, "w") as f: json.dump(result, f, indent=2)
    env.upload_to_hf(res_file, env.hf_results_repo, f"results/ablation/{name}.json")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 5: Summary Tables
# ═══════════════════════════════════════════════════════════
def print_table(tbl_num, title, filter_fn):
    print(f"\n{'='*72}\nTABLE {tbl_num}: {title}\n{'='*72}")
    print(f"{'Configuration':<36} {'Params':>8} {'Val Acc':>8} {'Test Acc':>8}\n{'-'*72}")
    rows = []
    for r in all_results:
        if filter_fn(r):
            print(f"{r['label']:<36} {r['params']:>8} {r['best_val_acc']:>7.2f}% {r['test_acc']:>7.2f}%")
            rows.append(r)
    
    df = pd.DataFrame(rows)[['name', 'label', 'params', 'best_val_acc', 'test_acc']]
    path = RESULTS_DIR / f"table_{tbl_num}_summary.csv"
    df.to_csv(path, index=False)
    env.upload_to_hf(path, env.hf_results_repo, f"results/ablation/table_{tbl_num}_summary.csv")

print_table(4, "Ablation of Frequency Components", lambda r: r.get('table') == 4)
print_table(5, "Ablation of Band Configurations", lambda r: r.get('table') == 5)
print_table(6, "MFFT Architecture Upgrades", lambda r: r.get('table') == 6)

print("\n✅ Ablation Study complete. Proceed to 06_paper_evals.ipynb")